# Heart Failure Prediction - Complete ML Pipeline

## Complete ML Workflow

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv('heart.csv')
print('Dataset Shape:', df.shape)
print('\nDataset Info:')
print(df.info())
print('\nTarget Distribution:')
print(df['HeartDisease'].value_counts())

## Step 3: Data Preprocessing

In [ ]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print('Categorical:', categorical_cols)
print('Numeric:', numeric_cols)

In [ ]:
# Encode categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
print('Categorical features encoded')

In [ ]:
# Handle zero values
for col in ['Cholesterol', 'RestingBP']:
    median_val = X[X[col] != 0][col].median()
    X[col] = X[col].replace(0, median_val)
print('Zero values replaced with median')

In [ ]:
# Scale numeric features
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
print('Features scaled successfully')

## Step 4: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test: {X_test.shape[0]} samples')

## Step 5: Train Multiple Models

In [ ]:
models = {
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'LDA': LinearDiscriminantAnalysis()
}
print('5 models will be trained with 5-fold cross-validation')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
test_results = {}
scoring = ['accuracy', 'roc_auc', 'recall', 'precision', 'f1']
for model_name, model in models.items():
    print(f'Training {model_name}...')
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    cv_results[model_name] = {'ROC-AUC': cv_scores['test_roc_auc'].mean()}
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    test_results[model_name] = {'Test_ROC_AUC': roc_auc_score(y_test, model.predict(X_test))}
    print(f'  CV ROC-AUC: {cv_results[model_name]["ROC-AUC"]:.4f}')

## Step 6: Model Comparison & Selection

In [ ]:
cv_df = pd.DataFrame(cv_results).T
test_df = pd.DataFrame(test_results).T
print('Cross-Validation Results:')
print(cv_df.sort_values('ROC-AUC', ascending=False))
best_model_name = cv_df['ROC-AUC'].idxmax()
print(f'\nBest Model: {best_model_name}')

## Step 7: Final Results

In [ ]:
print('PIPELINE SUMMARY')
print('='*50)
print(f'Dataset: 918 samples, 11 features')
print(f'Models Tested: 5')
print(f'Best Model: {best_model_name}')
print(f'CV ROC-AUC: {cv_df.loc[best_model_name, "ROC-AUC"]:.4f}')
print(f'Status: Ready for Deployment!')